# Model 5: extend_a_method (RoBERTa + Linguistic Features)

## 
This notebook is dedicated to the Part C extension method and keeps experimental settings comparable with teammate models.




## 1. Comparable Setup

- Seed: `42`
- Split: `train/val/test = 8/1/1` (shared split)
- Columns: `text`, `label` (`0=human`, `1=AI-generated`)
- Max length: `256`
- Training budget: `batch_size=16`, `epochs=3`, `lr=2e-5`
- Metrics: Accuracy, Precision, Recall, F1, Macro-F1


In [1]:
# If needed, install packages once
# !pip install -q transformers datasets scikit-learn nltk

import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cuda


In [2]:
import torch
print(torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))


2.9.1+cu128
cuda available: True
gpu count: 1
gpu name: NVIDIA GeForce RTX 5080


## 2. Data Loading

Replace paths below with your team's shared split files.


In [3]:
# TODO: replace with your actual split files
TRAIN_PATH = 'data/train.csv'
VAL_PATH = 'data/val.csv'
TEST_PATH = 'data/test.csv'

def load_split(path):
    df = pd.read_csv(path)
    assert 'text' in df.columns and 'label' in df.columns, 'Need columns: text, label'
    return df[['text', 'label']].dropna().reset_index(drop=True)

# Uncomment after paths are ready
# train_df = load_split(TRAIN_PATH)
# val_df = load_split(VAL_PATH)
# test_df = load_split(TEST_PATH)
# print(len(train_df), len(val_df), len(test_df))


## 3. Linguistic Features


In [4]:
import string

SENT_RE = re.compile(r"[.!?]+")
WORD_RE = re.compile(r"\b\w+\b")
PUNCT_SET = set(string.punctuation)

STOPWORDS = set([
    'the','a','an','and','or','but','if','to','of','in','on','for','with','as','by','is','are','was','were','be','been','being'
])

def linguistic_features(text: str):
    text = str(text)
    words = WORD_RE.findall(text.lower())
    n_words = max(len(words), 1)
    n_chars = max(len(text), 1)

    unique_words = len(set(words))
    lexical_diversity = unique_words / n_words

    sents = [s for s in SENT_RE.split(text) if s.strip()]
    avg_sentence_len = n_words / max(len(sents), 1)

    punct_ratio = sum(ch in PUNCT_SET for ch in text) / n_chars
    stopword_ratio = sum(w in STOPWORDS for w in words) / n_words
    avg_word_len = sum(len(w) for w in words) / n_words
    digit_ratio = sum(ch.isdigit() for ch in text) / n_chars

    return np.array([
        lexical_diversity,
        avg_sentence_len,
        punct_ratio,
        stopword_ratio,
        avg_word_len,
        digit_ratio,
    ], dtype=np.float32)


## 4. Extended Model

Fusion: `[CLS] embedding + linguistic features -> classifier`


In [5]:
class ExtendedRoBERTaClassifier(nn.Module):
    def __init__(self, base_model='roberta-base', feat_dim=6, hidden=256, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model)
        enc_dim = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(enc_dim + feat_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2)
        )

    def forward(self, input_ids, attention_mask, feats):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        x = torch.cat([cls, feats], dim=1)
        return self.classifier(x)


## 5. Training & Evaluation Skeleton


In [6]:
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    mp, mr, mf1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        'accuracy': acc,
        'precision': p,
        'recall': r,
        'f1': f1,
        'macro_f1': mf1,
        'macro_precision': mp,
        'macro_recall': mr,
    }

print('Ready: implement dataloader + training loop here based on your team pipeline.')


Ready: implement dataloader + training loop here based on your team pipeline.


## 6. Report Table

| Model | Extra Feature | Accuracy | Precision | Recall | Macro-F1 | Delta Macro-F1 vs RoBERTa |
|---|---|---:|---:|---:|---:|---:|
| TF-IDF + Logistic Regression | No |  |  |  |  |  |
| BERT-base | No |  |  |  |  |  |
| RoBERTa-base | No |  |  |  |  | 0.00 |
| RoBERTa-Extended (Ours) | Linguistic (6) |  |  |  |  | + |
| RoBERTa-Extended (Ablation) | Linguistic (partial) |  |  |  |  | + |


## 7. Build Dataset from HC3 all.jsonl

This section creates a binary dataset:
- label `0`: human-written
- label `1`: AI-generated


In [7]:
from pathlib import Path

CWD = Path.cwd().resolve()
raw_candidates = [CWD / 'data' / 'all.jsonl'] + [p / 'data' / 'all.jsonl' for p in CWD.parents]
RAW_JSONL = next((p for p in raw_candidates if p.exists()), None)
if RAW_JSONL is None:
    raise FileNotFoundError(f'Cannot locate data/all.jsonl from {CWD}')

DATA_DIR = RAW_JSONL.parent
SPLIT_DIR = DATA_DIR / 'splits'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print('CWD =', CWD)
print('RAW_JSONL =', RAW_JSONL)
print('DATA_DIR =', DATA_DIR)

import json
from sklearn.model_selection import train_test_split

RAW_JSONL = DATA_DIR / 'all.jsonl'
OUT_DIR = SPLIT_DIR

if not RAW_JSONL.exists():
    raise FileNotFoundError(f'Cannot find dataset: {RAW_JSONL}')


def flatten_hc3_record(obj):
    rows = []
    source = obj.get('source', 'unknown')

    for x in obj.get('human_answers', []) or []:
        if isinstance(x, str) and x.strip():
            rows.append({'text': x.strip(), 'label': 0, 'source': source})

    for x in obj.get('chatgpt_answers', []) or []:
        if isinstance(x, str) and x.strip():
            rows.append({'text': x.strip(), 'label': 1, 'source': source})

    return rows


all_rows = []
with RAW_JSONL.open('r', encoding='utf-8', errors='replace') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        all_rows.extend(flatten_hc3_record(obj))

full_df = pd.DataFrame(all_rows)
full_df = full_df.drop_duplicates(subset=['text', 'label']).reset_index(drop=True)

DEBUG_MAX = None
if DEBUG_MAX is not None and len(full_df) > DEBUG_MAX:
    full_df = full_df.sample(DEBUG_MAX, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    full_df,
    test_size=0.2,
    random_state=SEED,
    stratify=full_df['label']
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df['label']
)

train_df.to_csv(OUT_DIR / 'train.csv', index=False)
val_df.to_csv(OUT_DIR / 'val.csv', index=False)
test_df.to_csv(OUT_DIR / 'test.csv', index=False)

print('Saved split files to', OUT_DIR)
print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('label ratio train:', train_df['label'].value_counts(normalize=True).to_dict())






CWD = D:\6713\group assignment\comp6713-ass2\model04_extend_a_method
RAW_JSONL = D:\6713\group assignment\comp6713-ass2\data\all.jsonl
DATA_DIR = D:\6713\group assignment\comp6713-ass2\data


Saved split files to D:\6713\group assignment\comp6713-ass2\data\splits
train/val/test: 63464 7933 7934
label ratio train: {0: 0.6691667717130972, 1: 0.3308332282869028}


## 8. Dataset + Dataloader


In [8]:
from pathlib import Path

CWD = Path.cwd().resolve()
raw_candidates = [CWD / 'data' / 'all.jsonl'] + [p / 'data' / 'all.jsonl' for p in CWD.parents]
RAW_JSONL = next((p for p in raw_candidates if p.exists()), None)
if RAW_JSONL is None:
    raise FileNotFoundError(f'Cannot locate data/all.jsonl from {CWD}')

DATA_DIR = RAW_JSONL.parent
SPLIT_DIR = DATA_DIR / 'splits'
if not SPLIT_DIR.exists():
    raise FileNotFoundError(f'Cannot find split directory: {SPLIT_DIR}. Run Section 7 first.')

print('CWD =', CWD)
print('RAW_JSONL =', RAW_JSONL)
print('DATA_DIR =', DATA_DIR)

from torch.utils.data import Dataset, DataLoader

MODEL_NAME = 'roberta-base'
MAX_LENGTH = 256
BATCH_SIZE = 16


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ExtendedTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text'])
        label = int(row['label'])

        enc = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        feats = linguistic_features(text)

        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'feats': torch.tensor(feats, dtype=torch.float32),
            'labels': torch.tensor(label, dtype=torch.long),
        }


train_df = pd.read_csv(SPLIT_DIR / 'train.csv')
val_df = pd.read_csv(SPLIT_DIR / 'val.csv')
test_df = pd.read_csv(SPLIT_DIR / 'test.csv')

train_ds = ExtendedTextDataset(train_df, tokenizer, MAX_LENGTH)
val_ds = ExtendedTextDataset(val_df, tokenizer, MAX_LENGTH)
test_ds = ExtendedTextDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print('DataLoaders ready:', len(train_loader), len(val_loader), len(test_loader))






CWD = D:\6713\group assignment\comp6713-ass2\model04_extend_a_method
RAW_JSONL = D:\6713\group assignment\comp6713-ass2\data\all.jsonl
DATA_DIR = D:\6713\group assignment\comp6713-ass2\data


DataLoaders ready: 3967 496 496


## 9. Train / Validate / Test


In [9]:
from copy import deepcopy
from tqdm.auto import tqdm

LR = 2e-5
EPOCHS = 3
WEIGHT_DECAY = 0.01
PATIENCE = 2


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []
    criterion = nn.CrossEntropyLoss()

    for batch in tqdm(loader, leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        feats = batch['feats'].to(device)
        labels = batch['labels'].to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(input_ids=input_ids, attention_mask=attention_mask, feats=feats)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item())
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(labels.detach().cpu().tolist())

    metrics = compute_metrics(all_labels, all_preds)
    metrics['loss'] = total_loss / max(len(loader), 1)
    return metrics


model = ExtendedRoBERTaClassifier(base_model=MODEL_NAME).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_state = None
best_val = -1.0
wait = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(model, train_loader, optimizer=optimizer)
    val_m = run_epoch(model, val_loader, optimizer=None)

    history.append({'epoch': epoch, 'train': train_m, 'val': val_m})
    print(f"Epoch {epoch} | train_f1={train_m['f1']:.4f} val_macro_f1={val_m['macro_f1']:.4f}")

    if val_m['macro_f1'] > best_val:
        best_val = val_m['macro_f1']
        best_state = deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print('Early stopping triggered.')
            break

if best_state is not None:
    model.load_state_dict(best_state)

test_m = run_epoch(model, test_loader, optimizer=None)
print('Test metrics:', test_m)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/3967 [00:00<?, ?it/s]

  0%|          | 0/496 [00:00<?, ?it/s]

Epoch 1 | train_f1=0.9893 val_macro_f1=0.9973


...

  0%|          | 0/3967 [00:00<?, ?it/s]

  0%|          | 0/496 [00:00<?, ?it/s]

Epoch 3 | train_f1=0.9986 val_macro_f1=0.9942


  0%|          | 0/496 [00:00<?, ?it/s]

Test metrics: {'accuracy': 0.9993698008570708, 'precision': 0.9980988593155894, 'recall': 1.0, 'f1': 0.9990485252140818, 'macro_f1': 0.9992887024449755, 'macro_precision': 0.9990494296577948, 'macro_recall': 0.9995291015257111, 'loss': 0.003514956070291426}


## 10. Export Results for Report


In [10]:
from pathlib import Path

CWD = Path.cwd().resolve()
raw_candidates = [CWD / 'data' / 'all.jsonl'] + [p / 'data' / 'all.jsonl' for p in CWD.parents]
RAW_JSONL = next((p for p in raw_candidates if p.exists()), None)
if RAW_JSONL is None:
    raise FileNotFoundError(f'Cannot locate data/all.jsonl from {CWD}')

DATA_DIR = RAW_JSONL.parent
SPLIT_DIR = DATA_DIR / 'splits'
if not SPLIT_DIR.exists():
    raise FileNotFoundError(f'Cannot find split directory: {SPLIT_DIR}. Run Section 7 first.')

print('CWD =', CWD)
print('RAW_JSONL =', RAW_JSONL)
print('DATA_DIR =', DATA_DIR)

if 'test_m' not in globals():
    raise RuntimeError('`test_m` not found. Please run Section 9 first.')

out_csv = SPLIT_DIR / 'extended_results.csv'

result_row = {
    'Model': 'RoBERTa-Extended (Ours)',
    'Extra Feature': 'Linguistic (6)',
    'Accuracy': round(float(test_m['accuracy']), 4),
    'Precision': round(float(test_m['precision']), 4),
    'Recall': round(float(test_m['recall']), 4),
    'Macro-F1': round(float(test_m['macro_f1']), 4),
}

result_df = pd.DataFrame([result_row])
result_df.to_csv(out_csv, index=False)
print('Saved:', out_csv)
result_df






CWD = D:\6713\group assignment\comp6713-ass2\model04_extend_a_method
RAW_JSONL = D:\6713\group assignment\comp6713-ass2\data\all.jsonl
DATA_DIR = D:\6713\group assignment\comp6713-ass2\data
Saved: D:\6713\group assignment\comp6713-ass2\data\splits\extended_results.csv


,Model,Extra Feature,Accuracy,Precision,Recall,Macro-F1
0,RoBERTa-Extended (Ours),Linguistic (6),0.9994,0.9981,1.0,0.9993


## 12. Visualization

This section plots confusion matrix and AI-probability histogram on the test set.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

if 'model' not in globals() or 'test_loader' not in globals():
    raise RuntimeError('Please run Section 9 (Train / Validate / Test) first.')

model.eval()
y_true, y_pred, y_prob_ai = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        feats = batch['feats'].to(device)
        labels = batch['labels'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, feats=feats)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        y_prob_ai.extend(probs[:, 1].cpu().numpy().tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob_ai = np.array(y_prob_ai)

# 1) Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human (0)', 'AI (1)'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix - Model 4 Extended Method')
plt.tight_layout()
plt.show()

# 2) Histogram of AI probability by true class
fig, ax = plt.subplots(figsize=(7, 4))
bins = 30
ax.hist(y_prob_ai[y_true == 0], bins=bins, alpha=0.6, label='True Human (0)')
ax.hist(y_prob_ai[y_true == 1], bins=bins, alpha=0.6, label='True AI (1)')
ax.set_xlabel('Predicted Probability of AI class (label=1)')
ax.set_ylabel('Count')
ax.set_title('Histogram of Predicted AI Probability')
ax.legend()
plt.tight_layout()
plt.show()

print('Confusion matrix:\n', cm)
